# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/M-Sheheryar-khan/FlyRank-ML-Internship-Starter-Repo/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

1. Unit of analysis: One row is one content item, for one client, on one day — the grain of fact_content_daily_performance (report_date + client_hash_id + content_hash_id).

2. Table(s): fact_content_daily_performance (main), joined to dim_content (content metadata, age) and dim_clients (per-client history start dates, to check GSC/GA4 availability).

3. Time window: month=2026-03 — a mid-panel month. I don't use fact_content_daily_performance_sample because that table is the final month (June 2026), the natural outcome window of any past→future label; developing on it would mean peeking at the future I'm supposed to predict.

4. What I'd predict/rank: A proxy "declining" label — whether a content item's search impressions drop in the second half of the month vs. the first half — used to rank pages for refresh review (Lane 2: Refresh/Content Opportunity Scoring).

5. What I exclude: fact_content_query_90d for this notebook — its 90-day window overlaps recent months, and lining that up correctly with my label window is more than this notebook needs; I'll bring it in later once the window alignment is worked out.

In [7]:
%pip -q install duckdb

import os
from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')

import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
MAR = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"

con.sql(f"DESCRIBE SELECT * FROM {MAR}").df()

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


In [8]:
con.sql(f"""
    SELECT COUNT(*) AS n_rows, MIN(report_date) AS min_d, MAX(report_date) AS max_d
    FROM {MAR}
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,n_rows,min_d,max_d
0,9841378,2026-03-01,2026-03-31


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

###Feature:

gsc_impressions, gsc_clicks, gsc_avg_position — observed search performance
ga4_pageviews, ga4_sessions, ga4_engaged_sessions — observed engagement (only when ga4_data_available IS TRUE)
sessions_ai — AI-referred sessions
content age (from dim_content.content_created_at, joined)

###Label / proxy:

a "declining" flag built by comparing impressions in the first half vs. second half of the month — never fed back in as a feature

###Context:

report_date, client_hash_id, content_hash_id

###Excluded:

fact_content_query_90d — its window overlaps recent months; aligning it correctly is out of scope for this notebook
rows where ga4_data_available IS NOT TRUE — excluded from any GA4-based feature, since zeros there mean "not tracked yet," not "no engagement"

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [10]:
grain_check = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
    FROM {MAR}
    GROUP BY 1, 2, 3
    HAVING c > 1
    LIMIT 5
""").df()

print("Duplicate grain rows found:", len(grain_check))
grain_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate grain rows found: 0


,report_date,client_hash_id,content_hash_id,c


In [11]:
con.sql(f"""
    SELECT COUNT(*) AS n_rows, MIN(report_date) AS min_d, MAX(report_date) AS max_d,
           COUNT(DISTINCT content_hash_id) AS n_content,
           COUNT(DISTINCT client_hash_id) AS n_clients
    FROM {MAR}
""").df()

,n_rows,min_d,max_d,n_content,n_clients
0,9841378,2026-03-01,2026-03-31,331437,55


In [12]:
con.sql(f"""
    SELECT COUNT(*) AS total_rows,
           SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available_rows,
           ROUND(SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS pct_available
    FROM {MAR}
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,ga4_available_rows,pct_available
0,9841378,413966.0,4.21


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.